In [ ]:
from delta.tables import DeltaTable
import pyspark.sql.functions as F

StatementMeta(, , -1, SessionError, , SessionError, True)

InvalidHttpRequest: [TooManyRequestsForCapacity] [TooManyRequestsForCapacity] HTTP Response code 430: This Spark job can’t be run because you’ve hit a Spark compute or API rate limit. To proceed, cancel an active Spark job through the Monitoring hub, choose a larger capacity SKU, or try again later. For more visibility and control, go to Workspace settings → Job management (Job Concurrency & Queue Monitoring) to review running and queued Spark jobs, understand capacity contention, and take action as needed. [Learn more at 'https://go.microsoft.com/fwlink/?linkid=2356970&clcid=0x409']. HTTP status code: 430.

In [ ]:
%run /nb_common_utils

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
ctx = log_start(
    spark,
    run_id    = run_id,
    file_name = "stg_new_customers , stg_new_product, stg_transaction",
    entity    = "transaction",
    load_type = load_type
)

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
fact_table = "gold.fact_sale"
silver_target_tabale = "silver.clean_products"

df_stg_tx   = spark.table("stage.stg_new_transactions")
df_stg_prod = spark.table("stage.stg_new_products")
df_stg_cust = spark.table("stage.stg_new_customers")

In [ ]:
df_price = (
    spark.table(silver_target_tabale)
         .select("product_id", "unit_price")
         .alias("base")
    .join(
        df_stg_prod.select("product_id", "unit_price").alias("stg"),
        on="product_id",
        how="left"
    )
    .select(
        F.col("base.product_id"),
        F.coalesce(
            F.col("stg.unit_price"),
            F.col("base.unit_price")
        ).alias("unit_price")
    )
)

In [11]:
df_fact_new = (
    df_stg_tx
        .filter(F.col("change_type").isin("INSERT", "UPDATE", "UPSERT"))
        .alias("tx")
   
    .join(
        df_price.alias("p"),
        on="product_id",
        how="left"
    )
    
    .join(
        df_stg_cust.select("customer_id").alias("c"),
        on="customer_id",
        how="left"
    )
    .select(
        F.col("tx.order_id"),
        F.col("tx.customer_id"),
        F.col("tx.product_id"),
        F.col("tx.order_date"),
        F.col("tx.quantity"),
        F.col("p.unit_price"),
        F.col("tx.line_amount"),
        
        (F.col("tx.quantity") * F.col("p.unit_price"))
            .alias("total_amount"),
        F.col("tx.updated_at"),
        F.col("tx.change_type"),
    )
)

StatementMeta(, f75c6899-12b3-4ecc-bf33-abcbef9baedc, 13, Finished, Available, Finished, False)

In [12]:
df_fact_new.cache()
print(f"Rows to process: {df_fact_new.count()}")

StatementMeta(, f75c6899-12b3-4ecc-bf33-abcbef9baedc, 14, Finished, Available, Finished, False)

Rows to process: 500


In [16]:
# p_config_load_type = "INCREMENTAL"
# p_default_load_type = "FULL"


StatementMeta(, f75c6899-12b3-4ecc-bf33-abcbef9baedc, 18, Finished, Available, Finished, False)

In [17]:
load_type = (
    p_config_load_type.strip()
    if p_config_load_type and p_config_load_type.strip() != ""
    else p_default_load_type.strip()
)

print(f"Current load type: {load_type}")

StatementMeta(, f75c6899-12b3-4ecc-bf33-abcbef9baedc, 19, Finished, Available, Finished, False)

Current load type: INCREMENTAL


In [18]:
try:

    if load_type == "FULL":

        print("Running FULL LOAD — fact_sale...")

        df_full = df_fact_new.withColumn(
            "fact_id",
            F.monotonically_increasing_id()
        ).drop("change_type")

        (
            df_full.write
                .mode("overwrite")
                .format("delta")
                .saveAsTable(fact_table)
        )

        print(f"FULL LOAD completed: {df_full.count()} rows")

        log_end(spark, ctx, status="SUCCESS", row_read = df_full.count(), rows_inserted =df_full.count())


    elif load_type == "INCREMENTAL":
    
        print("Running INCREMENTAL LOAD")

        if not spark.catalog.tableExists(fact_table):
            
            print("fact_sale not found -> Initial load")
            
            df_init = df_fact_new.withColumn(
                "fact_id",
                F.monotonically_increasing_id()
            ).drop("change_type")

            (
                df_init.write
                    .mode("overwrite")
                    .format("delta")
                    .saveAsTable(fact_table)
            )

            print(f"Initial load completed: {df_init.count()} rows")
            
            log_end(spark, ctx, status="SUCCESS", row_read = df_full.count(), rows_inserted =df_full.count())
        
        else:

            # ── INSERT rows 
            df_insert = df_fact_new.filter(
                F.col("change_type").isin("INSERT", "UPSERT")
            )

            insert_count = df_insert.count()

            if insert_count > 0:
                # handle conflict fact_id by get max id first
                max_id = (
                    spark.table(fact_table)
                        .agg(F.max("fact_id"))
                        .collect()[0][0] or 0
                )

                df_insert_with_id = (
                    df_insert
                        .drop("change_type")
                        .withColumn(
                            "fact_id",
                            F.monotonically_increasing_id() + max_id + 1
                        )
                )

                (
                    df_insert_with_id.write
                        .mode("append")
                        .format("delta")
                        .saveAsTable(fact_table)
                )

                print(f"Inserted: {insert_count} rows")
            
            # ── UPDATE rows ──────────────────────────────
            
            
            df_update = df_fact_new.filter(
                F.col("change_type") == "UPDATE"
            )

            update_count = df_update.count()

            if update_count > 0:

                delta_fact = DeltaTable.forName(spark, fact_table)

                (
                    delta_fact.alias("t")
                    .merge(
                        df_update.alias("s"),
                        "t.order_id = s.order_id"
                    )
                    .whenMatchedUpdate(set={
                        "customer_id":  "s.customer_id",
                        "product_id":   "s.product_id",
                        "order_date":   "s.order_date",
                        "quantity":     "s.quantity",
                        "unit_price":   "s.unit_price",
                        "line_amount":  "s.line_amount",
                        "total_amount": "s.total_amount",
                        "updated_at":   "s.updated_at",
                    })
                    .execute()
                )

                print(f"Updated: {update_count} rows")

            if insert_count == 0 and update_count == 0:
                print("No changes to apply — fact_sale unchanged")
            
            log_end(spark, ctx, status="SUCCESS", row_read = insert_count + update_count, rows_inserted =insert_count , rows_updated=update_count  )

    else:
        raise Exception(f"Unsupported load_type: {load_type}")
    
    

except Exception as e:
    log_end(spark, ctx, status="FAILED", error_message=str(e))
    raise

finally:
    df_fact_new.unpersist()

StatementMeta(, f75c6899-12b3-4ecc-bf33-abcbef9baedc, 20, Finished, Available, Finished, False)

Running INCREMENTAL LOAD
Inserted: 500 rows


DataFrame[order_id: string, customer_id: string, product_id: string, order_date: date, quantity: int, unit_price: double, line_amount: double, total_amount: double, updated_at: timestamp, change_type: string]